# Arbitrary array geometry

`AntennaArray` takes explicit element positions, so it handles any layout:
thinned grids, circular rings, randomly placed elements.

In [ ]:
import numpy as np

from arraybeam import AntennaArray

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = 'svg'
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True

## A thinned rectangular grid

Start from an 8 x 8 half-wavelength grid and randomly remove a third of the
elements.

In [ ]:
rng = np.random.default_rng(2024)

grid_x, grid_y = np.meshgrid(np.arange(8) * 0.5, np.arange(8) * 0.5)
keep = rng.random(grid_x.size) > 1 / 3

array = AntennaArray(x=grid_x.ravel()[keep], y=grid_y.ravel()[keep])
print(f'{array.num_elements} of 64 elements kept')

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(array.x, array.y)
plt.xlabel(r'Horizontal x ($\lambda$)')
plt.ylabel(r'Vertical y ($\lambda$)')
plt.title('Thinned array layout')
plt.axis('equal')
plt.show()

## Pattern

`get_pattern` sums directly over the element positions, so it does not care
that the layout is irregular.

In [ ]:
azimuth = np.arange(-90, 90, 0.5)
elevation = np.arange(-90, 90, 0.5)

result = array.get_pattern(azimuth, elevation, beam_az=20, beam_el=-10)
pattern_db = 20 * np.log10(np.abs(result['array_factor']) + 1e-8)

plt.figure(figsize=(7, 5))
plt.pcolormesh(elevation, azimuth, pattern_db,
               vmin=-40, vmax=0, shading='auto', cmap='jet')
plt.xlabel('Elevation (deg)')
plt.ylabel('Azimuth (deg)')
plt.title('Thinned array factor (dB)')
plt.colorbar(label='dB')
plt.show()

## A circular ring

A ring of 24 elements on a 2-wavelength radius.

In [ ]:
angles = np.arange(24) / 24 * 2 * np.pi
ring = AntennaArray(x=2.0 * np.cos(angles), y=2.0 * np.sin(angles))

plt.figure(figsize=(5, 5))
plt.scatter(ring.x, ring.y)
plt.xlabel(r'Horizontal x ($\lambda$)')
plt.ylabel(r'Vertical y ($\lambda$)')
plt.title('Circular array layout')
plt.axis('equal')
plt.show()

In [ ]:
result = ring.get_pattern(azimuth, elevation, beam_az=0, beam_el=0)
pattern_db = 20 * np.log10(np.abs(result['array_factor']) + 1e-8)

plt.figure(figsize=(7, 5))
plt.pcolormesh(elevation, azimuth, pattern_db,
               vmin=-40, vmax=0, shading='auto', cmap='jet')
plt.xlabel('Elevation (deg)')
plt.ylabel('Azimuth (deg)')
plt.title('Circular array factor (dB)')
plt.colorbar(label='dB')
plt.show()

## Supplying weights directly

`steering_weights` builds the complex weights that `get_pattern` would use
internally, so they can be inspected or modified first. Passing them back as
`weight` uses them verbatim, with no renormalisation.

Because every class in the package shares one sign convention, a `weight`
vector produced anywhere is accepted anywhere.

In [ ]:
weight = ring.steering_weights(beam_az=30, beam_el=0)

# Perturb the phases to emulate calibration error
noisy = weight * np.exp(1j * rng.normal(0, 0.4, weight.size))

for label, w in [('ideal', weight), ('0.4 rad rms phase error', noisy)]:
    result = ring.get_pattern(azimuth, elevation=0.0, weight=w)
    plt.plot(azimuth,
             20 * np.log10(np.abs(result['array_factor'])),
             label=label)

plt.xlabel('Azimuth (deg)')
plt.ylabel('Normalized amplitude (dB)')
plt.ylim(-40, 5)
plt.title('Effect of random phase error')
plt.legend()
plt.show()